<a href="https://colab.research.google.com/github/SantiCeballosM/mineria-datos-cine-exito/blob/main/Cine_Exito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto de Minería de Datos - Cine Éxito

## Pregunta objetivo

¿Qué características de las películas se asocian a un mayor éxito en taquilla, de manera que los administradores del cine puedan decidir qué títulos programar para asegurar una alta demanda?

## Fuentes

1. IMDb Dataset
2. TMDB Movies Dataset

## Integrantes

- Santiago Ceballos
- Jhon Harold Ramirez
- Juan Esteban Ceballos
- Erik Torres

Este Notebook corresponde a la etapa de adquisición y preparación inicial de los datos.

## 1. Importación de librerías

Se comienza importando las librerías necesarias donde se descarguen y localicen los datos para leerlos y preparar las fuentes de datos.
Principalmente se usa “Pandas” para manipular los datasets, también se usa “os” el cual ayuda a localizar archivos dentro de las carpetas y “Kagglehub” para obtener las fuentes.
Este Notebook está hecho para ejecutarse desde el principio en Google Colab.


In [1]:

# 1. IMPORTACIÓN DE LIBRERÍAS


import pandas as pd
import numpy as np
import os
import kagglehub

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 2. Descarga y localización de la fuente IMDb

Se descarga el dataset de IMDb mediante `kagglehub`.
En lugar de utilizar rutas como `/kaggle/input/...`, se utiliza la ruta que devuelve `kagglehub` y se buscan automáticamente los archivos requeridos.
Esto permite que el Notebook funcione en Google Colab sin depender de una ruta específica de Kaggle.


In [2]:

# 2. DESCARGA Y LOCALIZACIÓN DE IMDb


ruta_imdb = kagglehub.dataset_download("ashirwadsangwan/imdb-dataset")

print("Ruta de descarga de IMDb:")
print(ruta_imdb)

# Buscar automáticamente los archivos necesarios dentro
# de la carpeta descargada.
ruta_imdb_basics = None
ruta_imdb_ratings = None

for raiz, carpetas, archivos in os.walk(ruta_imdb):
    for archivo in archivos:
        if archivo == "title.basics.tsv":
            ruta_imdb_basics = os.path.join(raiz, archivo)
        elif archivo == "title.ratings.tsv":
            ruta_imdb_ratings = os.path.join(raiz, archivo)

if ruta_imdb_basics is None:
    raise FileNotFoundError(
        "No se encontró title.basics.tsv en el dataset IMDb descargado."
    )

if ruta_imdb_ratings is None:
    raise FileNotFoundError(
        "No se encontró title.ratings.tsv en el dataset IMDb descargado."
    )

print("\nArchivo IMDb Basics encontrado:")
print(ruta_imdb_basics)

print("\nArchivo IMDb Ratings encontrado:")
print(ruta_imdb_ratings)

100%|██████████| 1.71G/1.71G [00:14<00:00, 123MB/s] 


Extracting files...
Ruta de descarga de IMDb:
/root/.cache/kagglehub/datasets/ashirwadsangwan/imdb-dataset/versions/905

Archivo IMDb Basics encontrado:
/root/.cache/kagglehub/datasets/ashirwadsangwan/imdb-dataset/versions/905/title.basics.tsv

Archivo IMDb Ratings encontrado:
/root/.cache/kagglehub/datasets/ashirwadsangwan/imdb-dataset/versions/905/title.ratings.tsv


## 3. Revisión de la estructura de IMDb
Antes de proceder a cargar los datasets se revisa los encabezados para comprobar que columnas hay disponibles, a su vez se carga una muestra de “title.basics” para poder observar una estructura inicial de los registros.


In [3]:

# 3. REVISIÓN DE LA ESTRUCTURA DE IMDb


imdb_basics_header = pd.read_csv(
    ruta_imdb_basics,
    sep="\t",
    nrows=0
)

imdb_ratings_header = pd.read_csv(
    ruta_imdb_ratings,
    sep="\t",
    nrows=0
)

print("COLUMNAS IMDb - title.basics")
print(imdb_basics_header.columns.tolist())

print("\nCOLUMNAS IMDb - title.ratings")
print(imdb_ratings_header.columns.tolist())

# Muestra inicial de title.basics
imdb_basics_muestra = pd.read_csv(
    ruta_imdb_basics,
    sep="\t",
    nrows=5,
    na_values="\\N"
)

print("\nMuestra inicial de IMDb title.basics:")
display(imdb_basics_muestra)

print("Dimensiones de la muestra:")
print(imdb_basics_muestra.shape)

COLUMNAS IMDb - title.basics
['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genres']

COLUMNAS IMDb - title.ratings
['tconst', 'averageRating', 'numVotes']

Muestra inicial de IMDb title.basics:


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,NaN,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,NaN,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,NaN,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,NaN,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,NaN,1,Short


Dimensiones de la muestra:
(5, 9)


## 4. Carga de la información básica de películas IMDb

Solo se selecciona las columnas necesarias de “title.basics”
El campo “tconst” será un identificador de cada título para luego usarlo al integrar la información de ratings.


In [4]:

# 4. CARGA DE IMDb TITLE.BASICS


columnas_imdb = [
    "tconst",
    "titleType",
    "primaryTitle",
    "startYear",
    "runtimeMinutes",
    "genres"
]

imdb_peliculas = pd.read_csv(
    ruta_imdb_basics,
    sep="\t",
    usecols=columnas_imdb,
    na_values="\\N"
)

print("Datos IMDb title.basics cargados correctamente.")
print("Filas:", len(imdb_peliculas))
print("Columnas:", imdb_peliculas.shape[1])

display(imdb_peliculas.head())

/tmp/ipykernel_404/3267500367.py:14: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  imdb_peliculas = pd.read_csv(


Datos IMDb title.basics cargados correctamente.
Filas: 12818444
Columnas: 6


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,1894.0,1.0,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,1892.0,5.0,"Animation,Short"
2,tt0000003,short,Poor Pierrot,1892.0,5.0,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,1892.0,12.0,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,1893.0,1.0,Short


## 5. Carga de ratings de IMDb

Cargamos el archivo “title.ratings.tsv” en donde están las calificaciones promedio y la cantidad de votos de las películas, luego usamos tconst como identificador para poder relacionar los datos con información básica de las películas

In [5]:

# 5. CARGA DE IMDb TITLE.RATINGS

imdb_ratings = pd.read_csv(
    ruta_imdb_ratings,
    sep="\t",
    usecols=["tconst", "averageRating", "numVotes"],
    na_values="\\N"
)

print("Datos IMDb ratings cargados correctamente.")
print("Filas:", len(imdb_ratings))
print("Columnas:", imdb_ratings.shape[1])

display(imdb_ratings.head())

print("Dimensiones de IMDb Ratings:", imdb_ratings.shape)

Datos IMDb ratings cargados correctamente.
Filas: 1714406
Columnas: 3


,tconst,averageRating,numVotes
0,tt0000001,5.7,2231
1,tt0000002,5.4,326
2,tt0000003,6.4,2386
3,tt0000004,5.0,202
4,tt0000005,6.2,3097


Dimensiones de IMDb Ratings: (1714406, 3)


## 6. Filtrado de películas en IMDb

En IMDb se manejan varios tipos de títulos aunque este proyecto esta enfocado en las peliculas y solo se tienen en cuenta los registros que están como movie en la columna “titleType”

In [6]:
# ============================================================
# 6. FILTRAR SOLAMENTE PELÍCULAS
# ============================================================

imdb_peliculas = imdb_peliculas[
    imdb_peliculas["titleType"] == "movie"
].copy()

print("Cantidad de películas IMDb:", len(imdb_peliculas))

display(imdb_peliculas.head())

Cantidad de películas IMDb: 757843


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres
8,tt0000009,movie,Miss Jerry,1894.0,45.0,Romance
144,tt0000147,movie,The Corbett-Fitzsimmons Fight,1897.0,100.0,"Documentary,News,Sport"
498,tt0000502,movie,Bohemios,1905.0,100.0,NaN
570,tt0000574,movie,The Story of the Kelly Gang,1906.0,70.0,"Action,Adventure,Biography"
587,tt0000591,movie,The Prodigal Son,1907.0,90.0,Drama


## 7. Integración de las tablas de IMDb

Se unen title.basics y title.ratings usando la columna tconst que tienen en común. Se hace un inner merge para quedarse solo con las películas que tienen información en las dos tablas y se compara la cantidad de filas antes y después de la unión para llevar un control del proceso

In [7]:
# ============================================================
# 7. CRUCE IMDb BASICS + IMDb RATINGS
# ============================================================

filas_imdb_antes = len(imdb_peliculas)

imdb = pd.merge(
    imdb_peliculas,
    imdb_ratings,
    on="tconst",
    how="inner"
)

filas_imdb_despues = len(imdb)

print("Filas de películas antes:", filas_imdb_antes)
print("Filas después de agregar ratings:", filas_imdb_despues)

print("\nDimensiones del dataset IMDb integrado:", imdb.shape)

display(imdb.head())

Filas de películas antes: 757843
Filas después de agregar ratings: 349163

Dimensiones del dataset IMDb integrado: (349163, 8)


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0000009,movie,Miss Jerry,1894.0,45.0,Romance,5.3,239
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,1897.0,100.0,"Documentary,News,Sport",5.3,613
2,tt0000502,movie,Bohemios,1905.0,100.0,NaN,3.9,28
3,tt0000574,movie,The Story of the Kelly Gang,1906.0,70.0,"Action,Adventure,Biography",6.0,1092
4,tt0000591,movie,The Prodigal Son,1907.0,90.0,Drama,5.0,42


## 8. Descarga y localización de la fuente TMDB

Se descarga el dataset de TMDB usando kagglehub y se busca automáticamente el archivo TMDB_movie_dataset_v11.csv dentro de la carpeta descargada. Así se evita depender de una ruta específica que podría cambiar según la plataforma o el entorno donde se ejecute.

In [8]:
# ============================================================
# 8. DESCARGA Y LOCALIZACIÓN DE TMDB
# ============================================================

ruta_tmdb = kagglehub.dataset_download(
    "asaniczka/tmdb-movies-dataset-2023-930k-movies"
)

print("Ruta de descarga de TMDB:")
print(ruta_tmdb)

# Buscar automáticamente el archivo principal de TMDB.
ruta_tmdb_archivo = None

for raiz, carpetas, archivos in os.walk(ruta_tmdb):
    for archivo in archivos:
        if archivo == "TMDB_movie_dataset_v11.csv":
            ruta_tmdb_archivo = os.path.join(raiz, archivo)

if ruta_tmdb_archivo is None:
    raise FileNotFoundError(
        "No se encontró TMDB_movie_dataset_v11.csv "
        "en el dataset TMDB descargado."
    )

print("\nArchivo principal de TMDB encontrado:")
print(ruta_tmdb_archivo)

100%|██████████| 254M/254M [00:01<00:00, 162MB/s]

Extracting files...


Ruta de descarga de TMDB:
/root/.cache/kagglehub/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies/versions/1072

Archivo principal de TMDB encontrado:
/root/.cache/kagglehub/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies/versions/1072/TMDB_movie_dataset_v11.csv


## 9. Revisión de la estructura de TMDB

Antes de cargar los datos que se van a utilizar en el proyecto, se revisan los nombres de las columnas del archivo de TMDB para conocer su estructura.

In [9]:

# 9. REVISIÓN DE LA ESTRUCTURA DE TMDB


tmdb_header = pd.read_csv(
    ruta_tmdb_archivo,
    nrows=0
)

print("COLUMNAS TMDB")
print(tmdb_header.columns.tolist())

# Leer una pequeña muestra para comprobar la estructura.
tmdb_muestra = pd.read_csv(
    ruta_tmdb_archivo,
    nrows=5
)

print("\nMuestra inicial de TMDB:")
display(tmdb_muestra.head())

print("Dimensiones de la muestra:")
print(tmdb_muestra.shape)

COLUMNAS TMDB
['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'tagline', 'genres', 'production_companies', 'production_countries', 'spoken_languages', 'keywords']

Muestra inicial de TMDB:


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,Interstellar,The adventures of a group of explorers who mak...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,The Dark Knight,Batman raises the stakes in his war on crime. ...,130.643,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,Avatar,"In the 22nd century, a paraplegic Marine is di...",79.932,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,Enter the world of Pandora.,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,The Avengers,When an unexpected enemy emerges and threatens...,98.082,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,Some assembly required.,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com..."


Dimensiones de la muestra:
(5, 24)


## 10. Carga del dataset completo de TMDB

Se carga el dataset completo utilizando la ruta local después se seleccionarán las columnas principales necesarias para el cruce y para la etapa de análisis.

In [10]:
# ============================================================
# 10. CARGA DEL DATASET COMPLETO DE TMDB
# ============================================================

tmdb = pd.read_csv(ruta_tmdb_archivo)

print("Dataset TMDB cargado correctamente.")
print("Filas:", len(tmdb))
print("Columnas:", tmdb.shape[1])

display(tmdb.head())

Dataset TMDB cargado correctamente.
Filas: 1503896
Columnas: 24


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,Interstellar,The adventures of a group of explorers who mak...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,The Dark Knight,Batman raises the stakes in his war on crime. ...,130.643,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,Avatar,"In the 22nd century, a paraplegic Marine is di...",79.932,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,Enter the world of Pandora.,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,The Avengers,When an unexpected enemy emerges and threatens...,98.082,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,Some assembly required.,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com..."


## 11. Selección de columnas principales de TMDB

Se conservan las siguientes columnas relacionadas con identificación, títulos, calificaciones, votos, estado, fecha de lanzamiento, ingresos, duración, presupuesto, identificador IMDb, popularidad y géneros y La columna `imdb_id` va a ser usada  como llave para relacionar TMDB con IMDb.

In [11]:

# 11. SELECCIÓN DE COLUMNAS PRINCIPALES DE TMDB

columnas_tmdb = [
    "id",
    "title",
    "vote_average",
    "vote_count",
    "status",
    "release_date",
    "revenue",
    "runtime",
    "budget",
    "imdb_id",
    "popularity",
    "genres"
]

tmdb = tmdb[columnas_tmdb].copy()

print("Columnas seleccionadas de TMDB:")
print(tmdb.columns.tolist())

print("\nDimensiones de TMDB:")
print(tmdb.shape)

display(tmdb.head())

Columnas seleccionadas de TMDB:
['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'budget', 'imdb_id', 'popularity', 'genres']

Dimensiones de TMDB:
(1503896, 12)


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,popularity,genres
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,160000000,tt1375666,83.952,"Action, Science Fiction, Adventure"
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,165000000,tt0816692,140.241,"Adventure, Drama, Science Fiction"
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,185000000,tt0468569,130.643,"Drama, Action, Crime, Thriller"
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,237000000,tt0499549,79.932,"Action, Adventure, Fantasy, Science Fiction"
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,220000000,tt0848228,98.082,"Science Fiction, Action, Adventure"


## 12. Normalización de los identificadores

Antes del cruce principal se normalizan los identificadores `tconst` e `imdb_id` Se van conviertir  a tipo texto y se eliminan los espacios del  inicio y del final, Esto nos ayuda a  que ambas columnas tengan un formato compatible para realizar el `merge`.

In [12]:
# ============================================================
# 12. NORMALIZACIÓN DE IDENTIFICADORES
# ============================================================

imdb["tconst"] = imdb["tconst"].astype("string").str.strip()
tmdb["imdb_id"] = tmdb["imdb_id"].astype("string").str.strip()

print("Identificadores normalizados correctamente.")

Identificadores normalizados correctamente.


## 13. Verificación de los datos antes del cruce IMDb + TMDB

Antes de realizar el cruce se registra la cantidad de filas de cada fuente y También se verifican cuántos registros tiene disponible el identificador que va a ser  utilizado como la llave de integración.

In [13]:

# 13. VERIFICACIÓN ANTES DEL CRUCE IMDb + TMDB


filas_imdb = len(imdb)
filas_tmdb = len(tmdb)

print("====================================")
print("DATOS ANTES DEL CRUCE")
print("====================================")

print("Filas IMDb:", filas_imdb)
print("Filas TMDB:", filas_tmdb)

print(
    "\nIdentificadores IMDb disponibles:",
    imdb["tconst"].notna().sum()
)

print(
    "Identificadores TMDB disponibles:",
    tmdb["imdb_id"].notna().sum()
)

DATOS ANTES DEL CRUCE
Filas IMDb: 349163
Filas TMDB: 1503896

Identificadores IMDb disponibles: 349163
Identificadores TMDB disponibles: 687742


## 14. Cruce principal entre IMDb y TMDB

Se realiza el cruce de las dos fuentes principales mediante `pd.merge()`, La columna `tconst` de IMDb se relaciona con `imdb_id` de TMDB ya que ambas representan el identificador IMDb de la película y Se utiliza un `inner merge` para que conserve solo  los registros que tienen correspondencia en las dos fuentes.

In [14]:
# ============================================================
# 14. CRUCE PRINCIPAL IMDb + TMDB
# ============================================================

peliculas_cruzadas = pd.merge(
    imdb,
    tmdb,
    left_on="tconst",
    right_on="imdb_id",
    how="inner",
    suffixes=("_imdb", "_tmdb")
)

print("Cruce IMDb + TMDB realizado correctamente.")

Cruce IMDb + TMDB realizado correctamente.


## 15. Verificación del resultado del cruce

Después del `merge` se verifican nuevamente el número de filas y columnas del dataset resultante Esta comparación nos permite documentar cuántos registros teníamos de cada fuente antes del cruce y cuántas películas quedaron disponibles después de integrar las dos fuentes.


In [15]:

# 15. VERIFICACIÓN DESPUÉS DEL CRUCE

filas_cruzadas = len(peliculas_cruzadas)

print("====================================")
print("RESULTADO DEL CRUCE")
print("====================================")

print("Filas IMDb antes:", filas_imdb)
print("Filas TMDB antes:", filas_tmdb)
print("Filas después del cruce:", filas_cruzadas)

print(
    "Columnas después del cruce:",
    peliculas_cruzadas.shape[1]
)

print("\nDimensiones finales:")
print(peliculas_cruzadas.shape)

display(peliculas_cruzadas.head())

print("\nColumnas del dataset cruzado:")
print(peliculas_cruzadas.columns.tolist())

RESULTADO DEL CRUCE
Filas IMDb antes: 349163
Filas TMDB antes: 1503896
Filas después del cruce: 286497
Columnas después del cruce: 20

Dimensiones finales:
(286497, 20)


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres_imdb,averageRating,numVotes,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,popularity,genres_tmdb
0,tt0000009,movie,Miss Jerry,1894.0,45.0,Romance,5.3,239,356151,Miss Jerry,5.000,3,Released,1894-10-08,0,45,0,tt0000009,0.748,Romance
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,1897.0,100.0,"Documentary,News,Sport",5.3,613,147610,The Corbett-Fitzsimmons Fight,5.500,20,Released,1897-01-01,0,19,0,tt0000147,1.250,Documentary
2,tt0000574,movie,The Story of the Kelly Gang,1906.0,70.0,"Action,Adventure,Biography",6.0,1092,20105,The Story of the Kelly Gang,5.397,29,Released,1906-12-26,0,70,0,tt0000574,2.160,"Crime, Drama, Action, History"
3,tt0000591,movie,The Prodigal Son,1907.0,90.0,Drama,5.0,42,396922,The Prodigal Son,0.000,0,Released,1907-06-19,0,90,0,tt0000591,0.600,Drama
4,tt0000615,movie,Robbery Under Arms,1907.0,NaN,Drama,3.7,38,603786,Robbery Under Arms,1.000,1,Released,1907-10-01,0,60,0,tt0000615,0.600,Drama



Columnas del dataset cruzado:
['tconst', 'titleType', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres_imdb', 'averageRating', 'numVotes', 'id', 'title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'budget', 'imdb_id', 'popularity', 'genres_tmdb']


#16. Diccionario de datos

Se construye el diccionario de datos del dataset ya cruzado y terminado.
El diccionario documenta las siguientes columnas:
Nombre de la columna.
Tipo de dato.
Significado.
Fuente de origen.
Como IMDb y TMDB tienen una columna `genres`, pandas la distingue después del cruce como `genres_imdb` y `genres_tmdb` y asi termina este.


In [16]:

# 16. DICCIONARIO DE DATOS


descripcion_columnas = {
    "tconst": "Identificador único de la película en IMDb",
    "titleType": "Tipo de título registrado en IMDb",
    "primaryTitle": "Título principal de la película en IMDb",
    "startYear": "Año de inicio o lanzamiento registrado en IMDb",
    "runtimeMinutes": "Duración de la película en minutos según IMDb",
    "genres_imdb": "Géneros registrados para la película en IMDb",
    "averageRating": "Calificación promedio de la película en IMDb",
    "numVotes": "Cantidad de votos registrados en IMDb",

    "id": "Identificador único de la película en TMDB",
    "title": "Título de la película en TMDB",
    "vote_average": "Calificación promedio registrada en TMDB",
    "vote_count": "Cantidad de votos registrados en TMDB",
    "status": "Estado de la película registrado en TMDB",
    "release_date": "Fecha de lanzamiento registrada en TMDB",
    "revenue": "Ingresos o recaudación registrada para la película en TMDB",
    "runtime": "Duración de la película en minutos según TMDB",
    "budget": "Presupuesto registrado para la película en TMDB",
    "imdb_id": "Identificador IMDb asociado dentro de TMDB",
    "popularity": "Indicador de popularidad registrado por TMDB",
    "genres_tmdb": "Géneros registrados para la película en TMDB"
}

fuente_columnas = {
    "tconst": "IMDb",
    "titleType": "IMDb",
    "primaryTitle": "IMDb",
    "startYear": "IMDb",
    "runtimeMinutes": "IMDb",
    "genres_imdb": "IMDb",
    "averageRating": "IMDb",
    "numVotes": "IMDb",

    "id": "TMDB",
    "title": "TMDB",
    "vote_average": "TMDB",
    "vote_count": "TMDB",
    "status": "TMDB",
    "release_date": "TMDB",
    "revenue": "TMDB",
    "runtime": "TMDB",
    "budget": "TMDB",
    "imdb_id": "TMDB",
    "popularity": "TMDB",
    "genres_tmdb": "TMDB"
}

tipos_datos = peliculas_cruzadas.dtypes.astype(str).to_dict()

filas_diccionario = []

for columna in peliculas_cruzadas.columns:
    filas_diccionario.append({
        "columna": columna,
        "tipo_de_dato": tipos_datos.get(columna, "No definido"),
        "significado": descripcion_columnas.get(
            columna,
            "Columna resultante del cruce de las fuentes"
        ),
        "fuente": fuente_columnas.get(
            columna,
            "Resultado del cruce"
        )
    })

diccionario_datos = pd.DataFrame(filas_diccionario)

display(diccionario_datos)

,columna,tipo_de_dato,significado,fuente
0,tconst,string,Identificador único de la película en IMDb,IMDb
1,titleType,object,Tipo de título registrado en IMDb,IMDb
2,primaryTitle,object,Título principal de la película en IMDb,IMDb
3,startYear,float64,Año de inicio o lanzamiento registrado en IMDb,IMDb
4,runtimeMinutes,object,Duración de la película en minutos según IMDb,IMDb
5,genres_imdb,object,Géneros registrados para la película en IMDb,IMDb
6,averageRating,float64,Calificación promedio de la película en IMDb,IMDb
7,numVotes,int64,Cantidad de votos registrados en IMDb,IMDb
8,id,int64,Identificador único de la película en TMDB,TMDB
9,title,object,Título de la película en TMDB,TMDB


#17. Validación del diccionario de datos

Se comprueba primero que todas las columnas existentes en el dataset cruzado estén documentadas en el diccionario para permitir que este no tenga menos registros que el resultante.


In [17]:

# 17. VALIDACIÓN DEL DICCIONARIO


cantidad_columnas = len(peliculas_cruzadas.columns)
cantidad_documentadas = len(diccionario_datos)

print("Cantidad de columnas del dataset cruzado:", cantidad_columnas)
print("Cantidad de columnas documentadas:", cantidad_documentadas)

if cantidad_columnas == cantidad_documentadas:
    print("✓ Todas las columnas están documentadas.")
else:
    print("⚠ Revisar el diccionario de datos.")

Cantidad de columnas del dataset cruzado: 20
Cantidad de columnas documentadas: 20
✓ Todas las columnas están documentadas.


## 18. Exportación de los resultados

Se guardan los resultados de esta etapa en dos archivos CSV
`peliculas_cruzadas.csv`: este es el  dataset resultante de la integración entre IMDb y TMDB.
`diccionario_datos.csv`: documentación de las columnas del dataset cruzado.
Estos archivos corresponden a los productos de la etapa de .


In [18]:
# ============================================================
# 18. GUARDAR DATASET CRUZADO
# ============================================================

peliculas_cruzadas.to_csv(
    "peliculas_cruzadas.csv",
    index=False
)

print("✓ Dataset cruzado guardado como:")
print("peliculas_cruzadas.csv")

✓ Dataset cruzado guardado como:
peliculas_cruzadas.csv


In [19]:
# ============================================================
# GUARDAR DICCIONARIO DE DATOS
# ============================================================

diccionario_datos.to_csv(
    "diccionario_datos.csv",
    index=False
)

print("✓ Diccionario guardado como:")
print("diccionario_datos.csv")

✓ Diccionario guardado como:
diccionario_datos.csv


#19. Conclusión de la etapa de adquisición

En esta etapa se realizó la descarga y preparacion de IMDb y TMDB.
Se cargaron los datos utilizando la libreria pandas y se filtraron los registros correspondientes a películas y se integraron las tablas de IMDb mediante el identificador de `tconst`.
Posteriormente se cargó TMDB y se realizó el cruce entre las dos fuentes utilizando `tconst` nuevamente de IMDb e `imdb_id` de TMDB como identificadores comunes.
Se verificó el número de filas antes y después del cruce y se construyó el diccionario de datos con el nombre, tipo de dato, significado y también fuente de las columnas del dataset resultante.
El dataset cruzado obtenido constituye la base para las siguientes etapas del proyecto de minería de datos sobre las características relacionadas con el éxito de las películas.
